In [42]:
%load_ext dotenv
%dotenv

The dotenv extension is already loaded. To reload it, use:
  %reload_ext dotenv


In [43]:
import re
from typing import List

import pdfplumber
import requests

from utils import chat, chunk_text, embed, num_tokens_from_string

In [44]:
stepback_system_message = """
You are an expert at world knowledge. Your task is to step back
and paraphrase a question to a more generic step-back question, which
is easier to answer. Here are a few examples

"input": "Could the members of The Police perform lawful arrests?"
"output": "what can the members of The Police do?"

"input": "Jan Sindel’s was born in what country?"
"output": "what is Jan Sindel’s personal history?"
"""


def generate_stepback(question: str):
    user_message = f"""{question}"""
    step_back_question = chat(
        messages=[
            {"role": "system", "content": stepback_system_message},
            {"role": "user", "content": user_message},
        ]
    )
    return step_back_question

In [46]:
question = "Which team did Thierry Audel play for from 2007 to 2008?"
step_back_question = generate_stepback(question)
print(f"Stepback results: {step_back_question}")

Stepback results: What teams did Thierry Audel play for during his career?


In [47]:
remote_pdf_url = "https://arxiv.org/pdf/1709.00666.pdf"
pdf_filename = "ch03-downloaded.pdf"

response = requests.get(remote_pdf_url)

if response.status_code == 200:
    with open(pdf_filename, "wb") as pdf_file:
        pdf_file.write(response.content)
else:
    print("Failed to download the PDF. Status code:", response.status_code)

In [48]:
text = ""

with pdfplumber.open(pdf_filename) as pdf:
    for page in pdf.pages:
        text += page.extract_text()

In [49]:
def split_text_by_titles(text):
    # A regular expression pattern for titles that
    # match lines starting with one or more digits, an optional uppercase letter,
    # followed by a dot, a space, and then up to 50 characters
    title_pattern = re.compile(r"(\n\d+[A-Z]?\. {1,3}.{0,60}\n)", re.DOTALL)
    titles = title_pattern.findall(text)
    # Split the text at these titles
    sections = re.split(title_pattern, text)
    sections_with_titles = []
    # Append the first section
    sections_with_titles.append(sections[0])
    # Iterate over the rest of sections
    for i in range(1, len(titles) + 1):
        section_text = sections[i * 2 - 1].strip() + "\n" + sections[i * 2].strip()
        sections_with_titles.append(section_text)

    return sections_with_titles


sections = split_text_by_titles(text)
print(f"Number of sections: {len(sections)}")

Number of sections: 9


In [50]:
for s in sections:
    print(num_tokens_from_string(s))

153
256
4156
556
2677
789
634
192
600


In [51]:
print(sections[1])

1. Introduction
Towards the end of the last century, Times Magazine asked some of the World’s leading
personalities to pick their choice for the person of the century. The magazine compiled a list 100 most
influential people of 20th century and the German born scientist Albert Einstein topped the list.
Einstein’s choice as the person of the century didn’t invoke any resentment, it was generally agreed
that 20th century is the age of Science and undoubtedly, Einstein’s contribution to Science, to the
understanding of the intricate laws of nature was unparalleled. He greatly influenced modern science;
altered our views on space‐time, matter and energy, gave new interpretation to gravity etc. The
enormous popularity he enjoyed during his lifetime and even now, is rare for any individual; religious
leader, politician, film star. Even a child knows his name, not to speak of adults.
However, while Einstein is known as a great theoretical physicist, few possibly knew that he
had more than 50 

In [52]:
parent_chunks = []
for s in sections:
    parent_chunks.extend(chunk_text(s, 1024, 40))

In [53]:
cypher_import_query = """
MERGE (pdf:PDF {id:$pdf_id})
MERGE (p:Parent {id:$pdf_id + '-' + $id})
SET p.text = $parent
MERGE (pdf)-[:HAS_PARENT]->(p)
WITH p, $children AS children, $embeddings as embeddings
UNWIND range(0, size(children) - 1) AS child_index
MERGE (c:Child {id: $pdf_id + '-' + $id + '-' + toString(child_index)})
SET c.text = children[child_index], c.embedding = embeddings[child_index]
MERGE (p)-[:HAS_CHILD]->(c);
"""

In [54]:
from neo4j import GraphDatabase
import os
import json
import hashlib

driver = GraphDatabase.driver("neo4j://localhost:7687", auth=("neo4j", "newpassword"))

EMBEDDING_CACHE_PATH = "embedding_cache.json"


def _chunk_key(text: str) -> str:
    return hashlib.sha256(text.encode("utf-8")).hexdigest()


def load_embedding_cache(path=EMBEDDING_CACHE_PATH):
    if os.path.exists(path):
        with open(path, "r", encoding="utf-8") as f:
            return json.load(f)
    return {}


def save_embedding_cache(cache, path=EMBEDDING_CACHE_PATH):
    with open(path, "w", encoding="utf-8") as f:
        json.dump(cache, f)


embedding_cache = load_embedding_cache()

for i, chunk in enumerate(parent_chunks):
    child_chunks = chunk_text(chunk, 500, 20)

    missing_chunks = []
    missing_keys = []
    for c in child_chunks:
        key = _chunk_key(c)
        if key not in embedding_cache:
            missing_chunks.append(c)
            missing_keys.append(key)

    if missing_chunks:
        missing_embeddings = embed(missing_chunks)
        assert len(missing_embeddings) == len(missing_keys)
        for key, emb in zip(missing_keys, missing_embeddings):
            embedding_cache[key] = emb
        save_embedding_cache(embedding_cache)

    embeddings = [embedding_cache[_chunk_key(c)] for c in child_chunks]

    # Add to neo4j
    driver.execute_query(
        cypher_import_query,
        id=str(i),
        pdf_id="1709.00666",
        parent=chunk,
        children=child_chunks,
        embeddings=embeddings,
    )

In [55]:
index_name = "parent"

# Determine embedding dimension dynamically from a sample vector
sample_embedding = embed(["hello"])[0]
vector_dimensions = len(sample_embedding)

# Use actual model-based dimension, e.g., 1024 in your case
print(f"Embedding dimension detected: {vector_dimensions}")

driver.execute_query(
    f"""
    CREATE VECTOR INDEX parent IF NOT EXISTS
    FOR (c:Child)
    ON (c.embedding)
    
    OPTIONS {{ indexConfig: 
    {{`vector.dimensions`: {vector_dimensions},
    `vector.similarity_function`: 'cosine'}}
    }}
    """
)


Embedding dimension detected: 1024


EagerResult(records=[], summary=<neo4j._work.summary.ResultSummary object at 0x0000029F8DF624B0>, keys=[])

In [56]:
retrieval_query = """
CALL db.index.vector.queryNodes($index_name, $k * 4, $question_embedding)
YIELD node, score
MATCH (node)<-[:HAS_CHILD]-(parent)
WITH parent, max(score) AS score
RETURN parent.text AS text, score
ORDER BY score DESC
LIMIT toInteger($k)
"""

In [57]:
def parent_retrieval(question: str, k: int = 4) -> List[str]:
    question_embedding = embed([question])[0]

    similar_records, _, _ = driver.execute_query(
        retrieval_query,
        question_embedding=question_embedding,
        k=k,
        index_name=index_name,
    )

    return [record["text"] for record in similar_records]

In [58]:
documents = parent_retrieval("Who was the Einsten's collaborator on sound reproduction system?")

for d in documents:
    print(d)
    print("=" * 20)

(not
related to his ongoing theoretical investigations) or took upon some practical problem. As shown in
Table. 2, Einstein was involved in three major inventions; (i) refrigeration system with Leo Szilard, (ii)
Sound reproduction system with Rudolf Goldschmidt and (iii) automatic camera with Gustav Bucky.
He also obtained a patent for a design of a blouse. It must also be mentioned that none of Einstein’s
inventions came to consumer market and with the exception of the refrigeration system, they are of
historical importance only. Below, his inventions are discussed in detail.
Einstein migrated to US and Goldschmidt to England, the
friendship continued. On January 10, 1934 Goldschmidt and Einstein patented an “Electromagnetic
sound reproduction apparatus.” The patent has a history. One of the acquaintance of Einstein, Olga
Eisner, a distinguished singer, became hard on hearing, a serious drawback for a singer. In 1928,
Einstein asked Goldschmidt’s assistance in developing a new type of

In [59]:
generate_stepback("Who was the Einsten's collaborator on sound reproduction system?")

"What was Einstein's involvement in sound reproduction systems?"

In [60]:
answer_system_message = "You're en Einstein expert, but can only use the provided documents to respond to the questions."


def generate_answer(question: str, documents: List[str]) -> str:
    user_message = f"""
    Use the following documents to answer the question that will follow:
    {documents}

    ---

    The question to answer using information only from the above documents: {question}
    """
    result = chat(
        messages=[
            {"role": "system", "content": answer_system_message},
            {"role": "user", "content": user_message},
        ]
    )
    print("Response:", result)

In [61]:
def rag_pipeline(question: str) -> str:
    stepback_prompt = generate_stepback(question)
    print(f"Stepback prompt: {stepback_prompt}")
    documents = parent_retrieval(stepback_prompt)
    answer = generate_answer(question, documents)
    return answer

In [62]:
import time
from typing import List

def chat_with_retry(messages: List[dict], max_retries: int = 3, delay: int = 2) -> str:
    """Chat function with retry logic for handling transient API errors."""
    for attempt in range(max_retries):
        try:
            return chat(messages)
        except Exception as e:
            if attempt < max_retries - 1:
                print(f"Attempt {attempt + 1} failed: {e}. Retrying in {delay}s...")
                time.sleep(delay)
            else:
                raise e

# Update the generate_stepback function to use retry
def generate_stepback(question: str):
    user_message = f"""{question}"""
    step_back_question = chat_with_retry(
        messages=[
            {"role": "system", "content": stepback_system_message},
            {"role": "user", "content": user_message},
        ]
    )
    return step_back_question

# Update the generate_answer function to use retry
def generate_answer(question: str, documents: List[str]) -> str:
    user_message = f"""
    Use the following documents to answer the question that will follow:
    {documents}

    ---

    The question to answer using information only from the above documents: {question}
    """
    result = chat_with_retry(
        messages=[
            {"role": "system", "content": answer_system_message},
            {"role": "user", "content": user_message},
        ]
    )
    print("Response:", result)

In [63]:
generate_answer("Who was the Einsten's collaborator on sound reproduction system?", documents)

Response: Einstein's collaborator on the sound reproduction system was Rudolf Goldschmidt.
